# Event Camera Subsampling Example

## Section 1: Import Required Libraries

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import h5py
import hdf5plugin

# Add project path to sys.path - use relative path from notebook location
project_path = Path.cwd()  # Get notebook working directory
if str(project_path) not in sys.path:
    sys.path.insert(0, str(project_path))

from datatransforms.event_transforms import (
    SpatialSubsampling,
    TemporalSubsampling,
    DropEventRandomly,
    SpatioTemporalFilteringSubsampling,
    TOS2DHarrisSubsampling,
)
from utils.data_utils import pyg2numpy_event_convertor, numpy2pyg_event_convertor, make_structured_array
from omegaconf import OmegaConf, DictConfig

print("Libraries imported successfully!")

# Define the event structured array format
events_struct = np.dtype(
    [("x", np.int16), ("y", np.int16), ("t", np.int64), ("p", bool)]
)

# Define save function for HDF5
def save_events_to_h5(data, output_path, method_name='events', image_height=180, image_width=240):
    """
    Save event data to HDF5 file format.
    
    Args:
        data: PyTorch Geometric Data object
        output_path: Path to save the HDF5 file
        method_name: Name/key for the dataset
        image_height: Image height
        image_width: Image width
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Convert to numpy structured array
    struct_array = pyg2numpy_event_convertor(data)
    
    # Save to HDF5 with proper structure
    with h5py.File(output_path, 'w') as f:
        # Create group for events
        events_group = f.create_group('events')
        events_group.create_dataset('x', data=struct_array['x'], dtype=np.int16, compression='gzip')
        events_group.create_dataset('y', data=struct_array['y'], dtype=np.int16, compression='gzip')
        events_group.create_dataset('t', data=struct_array['t'], dtype=np.int64, compression='gzip')
        events_group.create_dataset('p', data=struct_array['p'], dtype=bool, compression='gzip')
        
        # Store metadata
        f.attrs['num_events'] = struct_array.shape[0]
        f.attrs['image_height'] = image_height
        f.attrs['image_width'] = image_width
        f.attrs['method_name'] = method_name

## Section 2: Load Event Data (from DSEC)

Download a seq from DSEC dataset from URL, and load event data from h5 file.

In [ ]:
import urllib.request
import zipfile

# Configuration - use relative paths
notebook_dir = Path.cwd()
data_dir = notebook_dir / 'data' / 'dsec'
dataset_url = 'https://download.ifi.uzh.ch/rpg/DSEC/train/zurich_city_04_c/zurich_city_04_c_events_left.zip'
zip_filename = 'zurich_city_04_c_events_left.zip'
h5_filename = 'events.h5'
max_events = 1_000_000  # Limit to 1 million events for memory efficiency

# Create data directory if it doesn't exist
data_dir.mkdir(parents=True, exist_ok=True)

zip_path = data_dir / zip_filename
h5_path = data_dir / h5_filename

# Download if not already present
if not h5_path.exists():
    if not zip_path.exists():
        print(f"Downloading dataset from {dataset_url}...")
        urllib.request.urlretrieve(dataset_url, str(zip_path))
        print(f"Downloaded to {zip_path}")
    
    # Extract zip file
    print(f"Extracting {zip_filename}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    print("Extraction complete")

# Load events from HDF5 (events are stored under 'events' group)
print(f"Loading events from {h5_path}...")
with h5py.File(h5_path, 'r') as f:
    # Load only the first max_events to avoid memory issues
    events_x = f['events/x'][:max_events]
    events_y = f['events/y'][:max_events]
    events_t = f['events/t'][:max_events].astype(np.int64)
    events_p = f['events/p'][:max_events].astype(bool)
    total_events_in_file = f['events/x'].shape[0]

# Image dimensions from dataset
image_height = int(events_y.max()) + 1
image_width = int(events_x.max()) + 1

# Create structured array
sample_events = make_structured_array(events_x, events_y, events_t, events_p, dtype=events_struct)

# Convert to Data object
data = numpy2pyg_event_convertor(sample_events)

# Add dataset metadata for filter-based transforms
data.label = ['zurich_city_04_c']
data.file_id = 'events.h5'

print(f"\nLoaded DSEC event data:")
print(f"  Total events in file: {total_events_in_file:,}")
print(f"  Events loaded: {len(sample_events):,} (limited to {max_events:,} for memory efficiency)")
print(f"  Image resolution: {image_width} x {image_height}")
print(f"  X range: [{sample_events['x'].min()}, {sample_events['x'].max()}]")
print(f"  Y range: [{sample_events['y'].min()}, {sample_events['y'].max()}]")
print(f"  T range: [{sample_events['t'].min()}, {sample_events['t'].max()}] (µs)")
print(f"  Polarity distribution: {np.sum(sample_events['p'])} positive, {np.sum(~sample_events['p'])} negative")


## Section 3: Spatial Subsampling

In [ ]:
def apply_spatial_subsampling(data, h_ratio=2, v_ratio=2, h_offset=0, v_offset=0):
    """
    Apply spatial subsampling by keeping every n-th pixel in x and y directions.
    
    Args:
        data: PyTorch Geometric Data object
        h_ratio: Horizontal subsampling ratio (keep every h_ratio-th pixel)
        v_ratio: Vertical subsampling ratio (keep every v_ratio-th pixel)
        h_offset: Horizontal offset (0 <= h_offset < h_ratio)
        v_offset: Vertical offset (0 <= v_offset < v_ratio)
    
    Returns:
        Subsampled Data object and statistics
    """
    transform = SpatialSubsampling(
        subsampling_ratios=(h_ratio, v_ratio),
        subsampling_offsets=(h_offset, v_offset)
    )
    
    original_count = data.pos.shape[0]
    data_subsampled = transform(data.clone())
    final_count = data_subsampled.pos.shape[0]
    reduction = 100 * (1 - final_count / original_count)
    
    stats = {
        'original_count': original_count,
        'final_count': final_count,
        'reduction_percent': reduction,
        'h_ratio': h_ratio,
        'v_ratio': v_ratio
    }
    
    return data_subsampled, stats

# Test spatial subsampling
h_r, v_r = 2, 2
subsampled_spatial, stats_spatial = apply_spatial_subsampling(data, h_ratio=h_r, v_ratio=v_r)

print(f"Spatial Subsampling (h_ratio={h_r}, v_ratio={v_r}):")
print(f"  Original events: {stats_spatial['original_count']:,}")
print(f"  Subsampled events: {stats_spatial['final_count']:,}")
print(f"  Reduction: {stats_spatial['reduction_percent']:.2f}%")

# Save spatial subsampling results
subsampled_dir = data_dir / 'subsampled'
output_path_spatial = subsampled_dir / 'spatial_subsampling.h5'
save_events_to_h5(subsampled_spatial, output_path_spatial, 'spatial_subsampling', image_height, image_width)
print(f"Saved to {output_path_spatial.name}")

## Section 4: Temporal Subsampling

In [ ]:
def apply_temporal_subsampling(data, subsampling_ratio=2, window_size=1, fixed_interval=False):
    """
    Apply temporal subsampling.
    
    Args:
        data: PyTorch Geometric Data object
        subsampling_ratio: Keep one out of every subsampling_ratio temporal intervals
        window_size: Window size in milliseconds
        fixed_interval: If True, interval length is fixed and equals window_size
    
    Returns:
        Subsampled Data object and statistics
    """
    transform = TemporalSubsampling(
        subsampling_ratio=subsampling_ratio,
        window_size=window_size,
        fixed_interval=fixed_interval,
        time_offset_coefficient=0.0
    )
    
    original_count = data.pos.shape[0]
    data_subsampled = transform(data.clone())
    final_count = data_subsampled.pos.shape[0]
    reduction = 100 * (1 - final_count / original_count)
    
    stats = {
        'original_count': original_count,
        'final_count': final_count,
        'reduction_percent': reduction,
        'subsampling_ratio': subsampling_ratio,
        'window_size': window_size
    }
    
    return data_subsampled, stats

# Test temporal subsampling
t_ratio = 10
subsampled_temporal, stats_temporal = apply_temporal_subsampling(data, subsampling_ratio=t_ratio, window_size=1)

print(f"Temporal Subsampling (ratio={t_ratio}):")
print(f"  Original events: {stats_temporal['original_count']:,}")
print(f"  Subsampled events: {stats_temporal['final_count']:,}")
print(f"  Reduction: {stats_temporal['reduction_percent']:.2f}%")

# Save temporal subsampling results
output_path_temporal = subsampled_dir / 'temporal_subsampling.h5'
save_events_to_h5(subsampled_temporal, output_path_temporal, 'temporal_subsampling', image_height, image_width)
print(f"Saved to {output_path_temporal.name}")

## Section 5: Random Ratio Subsampling

In [ ]:
def apply_random_ratio_subsampling(data, probability=0.5):
    """
    Apply random subsampling: drop events with given probability.
    
    Args:
        data: PyTorch Geometric Data object
        probability: Probability of dropping each event
    
    Returns:
        Subsampled Data object and statistics
    """
    # Create a config object for DropEventRandomly
    cfg = DictConfig({'random_ratio_subsampling': probability})
    transform = DropEventRandomly(cfg)
    
    original_count = data.pos.shape[0]
    data_subsampled = transform(data.clone())
    final_count = data_subsampled.pos.shape[0]
    reduction = 100 * (1 - final_count / original_count)
    
    stats = {
        'original_count': original_count,
        'final_count': final_count,
        'reduction_percent': reduction,
        'drop_probability': probability
    }
    
    return data_subsampled, stats

# Test random ratio subsampling
drop_prob = 0.05
subsampled_random, stats_random = apply_random_ratio_subsampling(data, probability=drop_prob)

print(f"Random Ratio Subsampling (drop_probability={drop_prob}):")
print(f"  Original events: {stats_random['original_count']:,}")
print(f"  Subsampled events: {stats_random['final_count']:,}")
print(f"  Reduction: {stats_random['reduction_percent']:.2f}%")

# Save random subsampling results
output_path_random = subsampled_dir / 'random_ratio_subsampling.h5'
save_events_to_h5(subsampled_random, output_path_random, 'random_ratio_subsampling', image_height, image_width)
print(f"Saved to {output_path_random.name}")

## Section 6: Spatiotemporal Filtering (density-based) Subsampling

⚠️ **Note on Computation Time**: 
The spatiotemporal filtering method computes and caches filter values on first run, which may take longer. Subsequent runs on the same data will use cached values and be much faster. Filter values are saved to disk and reused across runs.

In [ ]:
def apply_spatiotemporal_filtering_subsampling(data, tau=30, filter_size=7, sampling_threshold=0.1, image_h=180, image_w=240):
    """
    Apply spatiotemporal filtering-based subsampling.
    Keeps events based on recursive temporal filtering and Gaussian spatial weighting.
    
    Args:
        data: PyTorch Geometric Data object with label and file_id attributes
        tau: Temporal constant (milliseconds)
        filter_size: Filter size (must be odd)
        sampling_threshold: Threshold for event selection
        image_h: Image height
        image_w: Image width
    
    Returns:
        Subsampled Data object and statistics
    """
    cfg_all = DictConfig({
        'dataset': {
            'image_resolution': [image_h, image_w],
            'name': 'dsec',
            'dataset_path': str(data_dir)
        },
        'transform': {
            'train': {
                'spatiotemporal_filtering_subsampling': {
                    'transform': True,
                    'tau': tau,
                    'filter_size': filter_size,
                    'sampling_threshold': sampling_threshold,
                    'normalization_length': None,
                    'mean_normalized': False
                }
            }
        }
    })
    
    cfg_dict = OmegaConf.to_object(cfg_all['transform']['train'])
    transform = SpatioTemporalFilteringSubsampling(cfg_all, cfg_dict)
    
    original_count = data.pos.shape[0]
    data_subsampled = transform(data.clone())
    final_count = data_subsampled.pos.shape[0]
    reduction = 100 * (1 - final_count / original_count)
    
    stats = {
        'original_count': original_count,
        'final_count': final_count,
        'reduction_percent': reduction,
        'tau': tau,
        'filter_size': filter_size,
        'sampling_threshold': sampling_threshold
    }
    
    return data_subsampled, stats, transform

# Test spatiotemporal filtering
print("Applying spatiotemporal filtering subsampling...")
subsampled_stf, stats_stf, transform_stf = apply_spatiotemporal_filtering_subsampling(
    data, tau=30, filter_size=7, sampling_threshold=0.1, 
    image_h=image_height, image_w=image_width
)

print(f"Spatiotemporal Filtering Subsampling:")
print(f"  Original events: {stats_stf['original_count']:,}")
print(f"  Subsampled events: {stats_stf['final_count']:,}")
print(f"  Reduction: {stats_stf['reduction_percent']:.2f}%")

# Save spatiotemporal filtering results
output_path_stf = subsampled_dir / 'spatiotemporal_filtering.h5'
save_events_to_h5(subsampled_stf, output_path_stf, 'spatiotemporal_filtering', image_height, image_width)
print(f"Saved to {output_path_stf.name}")

In [ ]:
# Analyze the relationship between sampling_threshold and subsampling ratio

thresholds = 10**np.arange(-2, np.log10(5), 0.2)  # Log scale from 0.01 to 5
subsampling_ratios = []

original_count = data.pos.shape[0]

# Test each threshold by modifying transform_stf.sampling_threshold
for threshold in thresholds:
    transform_stf.sampling_threshold = threshold
    subsampled = transform_stf(data.clone())
    final_count = subsampled.pos.shape[0]
    subsampling_ratio = final_count / original_count
    subsampling_ratios.append(subsampling_ratio)

# Plot results
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(thresholds, np.array(subsampling_ratios), 'o-', linewidth=2, markersize=8, color='steelblue')
ax.set_xlabel('Sampling Threshold', fontsize=12, fontweight='bold')
ax.set_ylabel('Subsampling Ratio (retained events / original events)', fontsize=12, fontweight='bold')
ax.set_title('Spatiotemporal Filtering: Subsampling Ratio vs Sampling Threshold', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, which='both')
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

# Print analysis table
print("\n" + "="*80)
print("Sampling Threshold Analysis for Spatiotemporal Filtering (Log Scale)")
print("="*80)
print(f"{'Threshold':<15} {'Subsampling Ratio':<25} {'Retained %':<15} {'Reduction %':<15}")
print("-"*80)
for i, threshold in enumerate(thresholds):
    retained_pct = subsampling_ratios[i] * 100
    reduction_pct = 100 - retained_pct
    print(f"{threshold:<15.4f} {subsampling_ratios[i]:<25.4f} {retained_pct:<15.2f} {reduction_pct:<15.2f}")
print("="*80)

## Section 7: ToS 2D Harris Subsampling

⚠️ **Note on Computation Time**: 
The Harris corner detection method also computes and caches TOS-Harris values on first run, which may take longer. Subsequent runs will use cached values. Harris values are saved to disk and reused across runs.

In [ ]:
def apply_tos2d_harris_subsampling(data, filter_size=7, TOS_T=14, Harris_block_size=2, Harris_ksize=3, Harris_k=0.04, 
                                   sampling_threshold=0.1, image_h=180, image_w=240):
    """
    Apply Time-of-Spike 2D Harris corner detection subsampling.
    Selects events at detected interest points.
    
    Args:
        data: PyTorch Geometric Data object with label and file_id attributes
        filter_size: Filter size (must be odd)
        TOS_T: Temporal window parameter (pixels)
        Harris_block_size: Block size for Harris detection
        Harris_ksize: Aperture parameter for Sobel derivative
        Harris_k: Harris corner detection parameter
        sampling_threshold: Threshold for event selection
        image_h: Image height
        image_w: Image width
    
    Returns:
        Subsampled Data object and statistics
    """
    cfg_all = DictConfig({
        'dataset': {
            'image_resolution': [image_h, image_w],
            'name': 'dsec',
            'dataset_path': str(data_dir)
        },
        'transform': {
            'train': {
                'tos_2DHarris_subsampling': {
                    'transform': True,
                    'filter_size': filter_size,
                    'TOS_T': TOS_T,
                    'Harris_block_size': Harris_block_size,
                    'Harris_ksize': Harris_ksize,
                    'Harris_k': Harris_k,
                    'sampling_threshold': sampling_threshold,
                    'normalization_length': None,
                    'mean_normalized': False
                }
            }
        }
    })
    
    cfg_dict = OmegaConf.to_object(cfg_all['transform']['train'])
    transform = TOS2DHarrisSubsampling(cfg_all, cfg_dict)
    
    original_count = data.pos.shape[0]
    data_subsampled = transform(data.clone())
    final_count = data_subsampled.pos.shape[0]
    reduction = 100 * (1 - final_count / original_count)
    
    stats = {
        'original_count': original_count,
        'final_count': final_count,
        'reduction_percent': reduction,
        'filter_size': filter_size,
        'TOS_T': TOS_T,
        'Harris_block_size': Harris_block_size
    }
    
    return data_subsampled, stats, transform

# Test ToS 2D Harris subsampling
print("Applying ToS 2D Harris subsampling...")
subsampled_harris, stats_harris, transform_harris = apply_tos2d_harris_subsampling(
    data, filter_size=7, TOS_T=14, Harris_block_size=2, Harris_ksize=3, Harris_k=0.04,
    sampling_threshold=1.0, image_h=image_height, image_w=image_width
)

print(f"ToS 2D Harris Subsampling:")
print(f"  Original events: {stats_harris['original_count']:,}")
print(f"  Subsampled events: {stats_harris['final_count']:,}")
print(f"  Reduction: {stats_harris['reduction_percent']:.2f}%")

# Save ToS 2D Harris results
output_path_harris = subsampled_dir / 'tos_2d_harris.h5'
save_events_to_h5(subsampled_harris, output_path_harris, 'tos_2d_harris', image_height, image_width)
print(f"Saved to {output_path_harris.name}")

In [ ]:
# Analyze the relationship between sampling_threshold and subsampling ratio for Harris

thresholds_harris = 10**np.arange(-1, 4, 0.2)  # Log scale from 0.1 to 10000
subsampling_ratios_harris = []

original_count = data.pos.shape[0]

# Test each threshold by modifying transform_harris.sampling_threshold
for threshold in thresholds_harris:
    transform_harris.sampling_threshold = threshold
    subsampled = transform_harris(data.clone())
    final_count = subsampled.pos.shape[0]
    subsampling_ratio = final_count / original_count
    subsampling_ratios_harris.append(subsampling_ratio)

# Plot results
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(thresholds_harris, np.array(subsampling_ratios_harris), 'o-', linewidth=2, markersize=8, color='darkgreen')
ax.set_xscale('log')
ax.set_xlabel('Sampling Threshold (log scale)', fontsize=12, fontweight='bold')
ax.set_ylabel('Subsampling Ratio (retained events / original events)', fontsize=12, fontweight='bold')
ax.set_title('ToS 2D Harris: Subsampling Ratio vs Sampling Threshold', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, which='both')
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

# Print analysis table
print("\n" + "="*80)
print("Sampling Threshold Analysis for ToS 2D Harris (Log Scale)")
print("="*80)
print(f"{'Threshold':<15} {'Subsampling Ratio':<25} {'Retained %':<15} {'Reduction %':<15}")
print("-"*80)
for i, threshold in enumerate(thresholds_harris):
    retained_pct = subsampling_ratios_harris[i] * 100
    reduction_pct = 100 - retained_pct
    print(f"{threshold:<15.4f} {subsampling_ratios_harris[i]:<25.4f} {retained_pct:<15.2f} {reduction_pct:<15.2f}")
print("="*80)